## Baseline Model

In [1]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, median_absolute_error, r2_score

### Sélection de variables

In [2]:
train_data = pd.read_csv("../data/processed_dataset.csv")
test_data = pd.read_parquet("../data/test_data.parquet")

In [3]:
X_train = train_data.drop(
    ["LoyerMensuel_Log1", "LoyerMensuel_BIF", "IdentifiantMaison"],
    axis=1).select_dtypes(include=["number"])
y_train = train_data["LoyerMensuel_Log1"]

### Transformation de données de test

In [4]:
# Les variables numériques
feature_cols = ["AgeMaison", "Quartier_Target", "Indicateur_Confort", "Chambres_par_Superficie", "LoyerMensuel_Log1"]

for col in ['Salon', 'SalleDeBainInterieure', 'Parking', 'Meuble', 'Jardin']:
    test_data[col + "_Bin"] = test_data[col].map({"Oui": 1, "Non": 0}).fillna(0).astype(int)

neighbourhood_encoder = joblib.load("neighbourhood_encoder.joblib") 
test_data["Quartier_Target"] = neighbourhood_encoder.transform(test_data[["Quartier"]])[:, 0]

test_data["LoyerMensuel_Log1"] = np.log1p(test_data["LoyerMensuel_BIF"])

# Création de variables utiles pour le test
# cols_confort = ['Salon_Bin', 'SalleDeBainInterieure_Bin', 'Parking_Bin', 'Meuble_Bin', 'Jardin_Bin']
# test_data['Indicateur_Confort'] = test_data[cols_confort].sum(axis=1)

# test_data['Chambres_par_Superficie'] = test_data['Chambres'] / (test_data['Superficie_m2'] + 0.1)

# Séparation de X_test et y_test
X_test = test_data.drop(
    ["LoyerMensuel_Log1", "LoyerMensuel_BIF", "IdentifiantMaison"],
    axis=1).select_dtypes(include=["number"])
y_test = test_data["LoyerMensuel_Log1"]

X_test

,Chambres,Superficie_m2,DistanceRoute_m,AgeMaison,Salon_Bin,SalleDeBainInterieure_Bin,Parking_Bin,Meuble_Bin,Jardin_Bin,Quartier_Target
0,1.0,46.0,154.0,20.0,1,1,0,0,0,0.010428
1,4.0,217.0,211.0,23.0,1,1,0,0,0,0.002128
2,4.0,181.0,64.0,24.0,1,1,1,0,0,0.002182
3,6.0,253.0,255.0,2.0,1,1,0,0,1,0.001636
4,3.0,162.0,29.0,34.0,1,1,1,1,0,0.001636
...,...,...,...,...,...,...,...,...,...,...
377,4.0,183.0,213.0,14.0,1,1,0,0,1,0.002163
378,6.0,171.0,250.0,34.0,1,1,0,0,0,0.001636
379,5.0,223.0,293.0,16.0,1,1,0,0,1,0.002061
380,6.0,287.0,24.0,30.0,1,1,1,0,1,0.001636


### Dummy Régression

In [5]:
model_dummy_mean = DummyRegressor(strategy="mean").fit(X_train, y_train)
model_dummy_median = DummyRegressor(strategy="median").fit(X_train, y_train)

y_predict_dummy_mean = model_dummy_mean.predict(y_test)
y_predict_dummy_median = model_dummy_median.predict(y_test)

print(f"y_predict_dummy_mean : {y_predict_dummy_mean}")
print()
print(f"y_predict_dummy_median : {y_predict_dummy_median}")

y_predict_dummy_mean : [13.84418581 13.84418581 13.84418581 13.84418581 13.84418581 13.84418581
 13.84418581 13.84418581 13.84418581 13.84418581 13.84418581 13.84418581
 13.84418581 13.84418581 13.84418581 13.84418581 13.84418581 13.84418581
 13.84418581 13.84418581 13.84418581 13.84418581 13.84418581 13.84418581
 13.84418581 13.84418581 13.84418581 13.84418581 13.84418581 13.84418581
 13.84418581 13.84418581 13.84418581 13.84418581 13.84418581 13.84418581
 13.84418581 13.84418581 13.84418581 13.84418581 13.84418581 13.84418581
 13.84418581 13.84418581 13.84418581 13.84418581 13.84418581 13.84418581
 13.84418581 13.84418581 13.84418581 13.84418581 13.84418581 13.84418581
 13.84418581 13.84418581 13.84418581 13.84418581 13.84418581 13.84418581
 13.84418581 13.84418581 13.84418581 13.84418581 13.84418581 13.84418581
 13.84418581 13.84418581 13.84418581 13.84418581 13.84418581 13.84418581
 13.84418581 13.84418581 13.84418581 13.84418581 13.84418581 13.84418581
 13.84418581 13.84418581 13.

### Régression Linéaire simple

In [6]:
model = LinearRegression().fit(X_train, y_train)
y_predict = model.predict(X_test)

y_predict_series = pd.Series(y_predict, index=y_test.index)

compareson = pd.DataFrame({
    "Réel": y_test,
    "Prédict": y_predict_series
})

print(compareson.head(3))

        Réel    Prédict
0  12.205733  12.590373
1  13.559070  13.901718
2  13.983435  13.876637


### Analyse des erreurs

In [7]:
print(f"LinearModel : {round(mean_squared_error(y_test, y_predict), 2)}")
print(f"dummy median : {round(median_absolute_error(y_test, y_predict_dummy_median), 2)}")
print(f"dummy mean : {round(mean_squared_error(y_test, y_predict_dummy_mean), 2)}")

print()

print(f"r2_Score dummpy mean : {r2_score(y_test, y_predict_dummy_mean)}")
print(f"r2_Score dummpy median : {r2_score(y_test, y_predict_dummy_median)}")
print(f"r2_Score LinearModel : {r2_score(y_test, y_predict)}")

LinearModel : 0.18
dummy median : 0.44
dummy mean : 0.39

r2_Score dummpy mean : 0.0
r2_Score dummpy median : -0.00433290382836482
r2_Score LinearModel : 0.5258727697450276
